# 04 — Feature Engineering

We transform raw lap-level data into a **driver × race feature matrix** that the prediction  
model in notebook 05 can consume.

| Feature Group | Features |
|---------------|----------|
| Pace | best lap, mean lap, median lap, std, consistency CV% |
| Tyre | degradation slope per stint, avg tyre life per stop |
| Strategy | pit stop count, compound changes, strategy chain |
| Sector | fastest S1/S2/S3, sum of best sectors |
| Race context | round number, wet race flag |
| Targets | `final_position`, `top_10`, `top_3`, `finished` |

> **Prerequisite:** Run `01_data_collection.ipynb` first.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats as sp_stats

sys.path.insert(0, os.path.join('..', 'src'))
from data_processing import (
    load_lap_data,
    create_driver_session_summary,
    build_target_labels,
    build_feature_matrix,
)

sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## 1 · Load Raw Data

In [ ]:
RAW_PATH  = os.path.join('..', 'data', 'raw',       'all_grand_prix_laps_2025.csv')
SAVE_PATH = os.path.join('..', 'data', 'processed', 'features_2025.csv')

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

df = load_lap_data(RAW_PATH)
print(f'Raw laps: {len(df):,}   Rounds: {df["RoundNumber"].nunique()}   Drivers: {df["Driver"].nunique()}')

## 2 · Base Summary (from data_processing.py)

The shared helper builds pace statistics and tyre strategy per driver per race.

In [ ]:
summary = create_driver_session_summary(df)
print(f'Summary rows: {len(summary)}')
summary.head(3)

## 3 · Tyre Degradation Slope

For each driver×race×stint we fit a linear regression of LapTime ~ TyreLife.  
The slope (s/lap) captures how quickly that driver's tyres dropped off.

In [ ]:
def compute_degradation_slope(group: pd.DataFrame) -> float:
    """Linear slope of LapTime vs TyreLife. Requires >= 4 data points."""
    sub = group.dropna(subset=['LapTime', 'TyreLife'])
    if len(sub) < 4:
        return np.nan
    slope, *_ = sp_stats.linregress(sub['TyreLife'], sub['LapTime'])
    return round(slope, 5)


clean_laps = df[
    df['LapTime'].notna() &
    df['IsAccurate'].eq(True) &
    df['Deleted'].eq(False) &
    df['PitInTime'].isna() &
    df['PitOutTime'].isna()
].copy()

deg_slopes = (
    clean_laps
    .groupby(['EventName', 'Driver', 'Stint'])
    .apply(compute_degradation_slope)
    .reset_index(name='deg_slope')
)

# Average slope across all stints per driver per race
avg_deg = (
    deg_slopes
    .groupby(['EventName', 'Driver'])['deg_slope']
    .mean()
    .reset_index(name='avg_deg_slope')
)

print(f'Degradation slopes computed: {len(avg_deg)}')
avg_deg.head(3)

## 4 · Position Gain / Loss

How many positions a driver gained or lost compared to lap 1 — a proxy for race craft.

In [ ]:
pos_change = (
    df.dropna(subset=['Position'])
    .sort_values(['EventName', 'Driver', 'LapNumber'])
    .groupby(['EventName', 'Driver'])
    .apply(lambda g: pd.Series({
        'pos_lap1':  g.iloc[0]['Position'],
        'pos_final': g.iloc[-1]['Position'],
    }))
    .reset_index()
)
pos_change['pos_gained'] = pos_change['pos_lap1'] - pos_change['pos_final']  # positive = moved forward
print(f'Position change rows: {len(pos_change)}')
pos_change.head(3)

## 5 · Wet Race Flag

Races where intermediate or wet tyres were used behave very differently.  
We flag them so the model can account for this.

In [ ]:
wet_events = (
    df[df['Compound'].isin(['INTERMEDIATE', 'WET'])]
    .groupby('EventName')
    .size()
    .reset_index(name='wet_laps')
)
wet_events['is_wet_race'] = 1
wet_events = wet_events[['EventName', 'is_wet_race']]
print(f'Races with wet/intermediate tyres: {len(wet_events)}')
wet_events

## 6 · Assemble Final Feature Matrix

In [ ]:
# Build target labels (final_position, top_10, top_3, finished)
labeled = build_target_labels(df, summary)

# Merge in additional features
features = (
    labeled
    .merge(avg_deg,    on=['EventName', 'Driver'], how='left')
    .merge(pos_change[['EventName', 'Driver', 'pos_lap1', 'pos_gained']],
           on=['EventName', 'Driver'], how='left')
    .merge(wet_events, on='EventName', how='left')
)
features['is_wet_race'] = features['is_wet_race'].fillna(0).astype(int)

print(f'Feature matrix shape: {features.shape}')
features.head(3)

## 7 · Feature Distributions

Quick visual check — are the features well-distributed or heavily skewed?

In [ ]:
numeric_features = [
    'best_lap_sec', 'average_lap_sec', 'std_lap_sec', 'consistency',
    'pit_stops', 'compound_changes', 'avg_deg_slope', 'pos_gained',
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    data = features[col].dropna()
    ax.hist(data, bins=30, color='#FF8000', edgecolor='white', linewidth=0.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Feature Distributions — 2025 F1 Feature Matrix', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8 · Correlation Matrix

In [ ]:
corr_cols = numeric_features + ['final_position']
corr = features[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax,
    vmin=-1, vmax=1,
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 9 · Save Processed Features

In [ ]:
features.to_csv(SAVE_PATH, index=False)
print(f'Saved: {SAVE_PATH}')
print(f'Shape: {features.shape}')
print(f'Columns: {features.columns.tolist()}')

---
## ✅ Summary

- Raw lap data → **driver × race feature matrix** with pace, strategy, degradation and position features
- Saved to `data/processed/features_2025.csv`

Proceed to **[05_race_result_prediction.ipynb](05_race_result_prediction.ipynb)**.